## Load PDFs and convert to chunks

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Embedding and Vector Store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        
        Args: 
             model_name (str): HuggingFace model name for sentence embeddings.
        """

        self.model = model_name
        self._load_model()

    def _load_model(self):              ## We use _ before the method name to indicate that this is a private method and cannot not be called from outside the class.
        """Load the Sentence Transformer model."""
        try:
            print(f"🔄 Loading embedding model: {self.model}...")
            self.model = SentenceTransformer(self.model)
            print(f"Model loaded successfully. Embeddinig dimenson: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initialse the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

🔄 Loading embedding model: all-MiniLM-L6-v2...
Model loaded successfully. Embeddinig dimenson: 384


In [8]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 8 PDF files to process

Processing: annurev-economics-081023-0246380-final.pdf
  ✓ Loaded 30 pages

Processing: A New Growth Strategy for Developing Nations (Rodrik and Stiglitz).pdf
  ✓ Loaded 24 pages

Processing: Ekonomista--Shared-Prosperity-in-a-Fractured-World.pdf
  ✓ Loaded 15 pages

Processing: On Productivism.pdf
  ✓ Loaded 20 pages

Processing: Rethinking Global Governance (Stiglitz and Rodrik).pdf
  ✓ Loaded 18 pages

Processing: ideas_interests-2024-05v3.pdf
  ✓ Loaded 67 pages

Processing: What the Mercantilists Got Right_0.pdf
  ✓ Loaded 19 pages

Processing: Servicing Development.pdf
  ✓ Loaded 13 pages

Total documents loaded: 206


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'dvipdfm; modified using iText 2.1.7 by 1T3XT', 'creator': 'LaTeX with hyperref package', 'creationdate': '2024-08-22T17:59:14+05:30', 'moddate': '2024-12-02T17:25:08+00:00', 'source': '../data/annurev-economics-081023-0246380-final.pdf', 'total_pages': 30, 'page': 0, 'page_label': '213', 'source_file': 'annurev-economics-081023-0246380-final.pdf', 'file_type': 'pdf'}, page_content='Downloaded from www.annualreviews.org.  Guest (guest) IP:  50.212.81.237 On: Mon, 02 Dec 2024 17:25:08\nEC16_Art09_Rodrik ARjats.cls August 22, 2024 17:41\nAnnual Review of Economics\nThe New Economics of\nIndustrial Policy\nRéka Juhász,1 Nathan Lane,2 and Dani Rodrik 3\n1V ancounver School of Economics, University of British Columbia, V ancouver,\nBritish Columbia, Canada\n2Department of Economics, University of Oxford, Oxford, United Kingdom\n3John F . Kennedy School of Government, Harvard University, Cambridge, Massachusetts, USA;\nemail: dani_rodrik@harvard.edu\nAnnu. Rev

In [10]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [11]:
chunks=split_documents(all_pdf_documents)
chunks

Split 206 documents into 768 chunks

Example chunk:
Content: Downloaded from www.annualreviews.org.  Guest (guest) IP:  50.212.81.237 On: Mon, 02 Dec 2024 17:25:08
EC16_Art09_Rodrik ARjats.cls August 22, 2024 17:41
Annual Review of Economics
The New Economics o...
Metadata: {'producer': 'dvipdfm; modified using iText 2.1.7 by 1T3XT', 'creator': 'LaTeX with hyperref package', 'creationdate': '2024-08-22T17:59:14+05:30', 'moddate': '2024-12-02T17:25:08+00:00', 'source': '../data/annurev-economics-081023-0246380-final.pdf', 'total_pages': 30, 'page': 0, 'page_label': '213', 'source_file': 'annurev-economics-081023-0246380-final.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'dvipdfm; modified using iText 2.1.7 by 1T3XT', 'creator': 'LaTeX with hyperref package', 'creationdate': '2024-08-22T17:59:14+05:30', 'moddate': '2024-12-02T17:25:08+00:00', 'source': '../data/annurev-economics-081023-0246380-final.pdf', 'total_pages': 30, 'page': 0, 'page_label': '213', 'source_file': 'annurev-economics-081023-0246380-final.pdf', 'file_type': 'pdf'}, page_content='Downloaded from www.annualreviews.org.  Guest (guest) IP:  50.212.81.237 On: Mon, 02 Dec 2024 17:25:08\nEC16_Art09_Rodrik ARjats.cls August 22, 2024 17:41\nAnnual Review of Economics\nThe New Economics of\nIndustrial Policy\nRéka Juhász,1 Nathan Lane,2 and Dani Rodrik 3\n1V ancounver School of Economics, University of British Columbia, V ancouver,\nBritish Columbia, Canada\n2Department of Economics, University of Oxford, Oxford, United Kingdom\n3John F . Kennedy School of Government, Harvard University, Cambridge, Massachusetts, USA;\nemail: dani_rodrik@harvard.edu\nAnnu. Rev

## Vector Store

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 768


## Convert Chunks to Embeddings

In [13]:
## Convert the text to embeddings
text= [doc.page_content for doc in chunks]

## Generate embeddings for the chunks
embeddings= embedding_manager.generate_embeddings(text)

## Store in vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 768 texts...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Generated embeddings with shape: (768, 384)
Adding 768 documents to vector store...
Successfully added 768 documents to vector store
Total documents in collection: 1536


## Retriever Pipeline from Vector Store

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [15]:
rag_retriever.retrieve("How can we develop nations with a new strategy")

Retrieving documents for query: 'How can we develop nations with a new strategy'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_202cde25_240',
  'content': 'A New Gr owth Strategy for Developing Nations  105\n 28 Under the old trade theory, picking winners was simple—identifying projects or sectors \nin which a country had a comparative advantage given its resources endowment (e.g. its \ncapital, human capital, or natural capital). In more dynamic contexts, this approach is \nless helpful; with factor flows, comparative advantage relates only to immobile factors, \nand a country with a skills shortage may be able to obtain the requisite skills from abroad. \nMore importantly, changing technologies; institutions; and more broadly, individual, \norganisational, and institutional learning mean that comparative advantages can change \nover time. Thus, a critical question facing any country as it embarks on structural change \nis what comparative advantages to acquire (Greenwald and Stiglitz, 2013).\n 29 For a discussion of conditionality, see Mazzucato and Rodrik (2023).',
  'metadata': {'source_file':

In [16]:
rag_retriever.retrieve("Tell me about The Manthan Project in Uttar Pradesh")

Retrieving documents for query: 'Tell me about The Manthan Project in Uttar Pradesh'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_93e5fb7a_730',
  'content': 'McKenzie\xa0( 2017) finds that the program attracted entrepre -\nneurs who invest in higher innovation, resulting in higher \nsales and profits, even during periods of economic crises \n(e.g., in 2016).\n• Higher job creation: the competition had a positive impact \non individuals starting new businesses as well as those \ngrowing their existing business. Notably, winning firms \nwere also more likely to have 10 or more workers. Overall, \nthe competition created 7027 jobs (McKenzie\xa02017 ).\n4.3   |   The Manthan Project in Uttar Pradesh, India\nIn 2012, the Manthan Project provided Accredited Social Health \nWorkers (popularly known as ASHA workers or ASHAs) in two \ndistricts of the state of Uttar Pradesh, India, a mobile phone- based \nmultimedia job aid called mSakhi.5 This project was initiated as \na collaboration between the Bill and Melinda Gates Foundation, \nQualcomm, IntraHealth International, and the Government of \nUttar Pradesh

## RAG Pipeline- VectorDB To LLM Output Generation

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [24]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [33]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [34]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


## Integration Vectordb Context pipeline With LLM output

In [35]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [36]:
answer=rag_simple("What The Manthan Project in Uttar Pradesh?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What The Manthan Project in Uttar Pradesh?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
The Manthan Project in Uttar Pradesh, India provided Accredited Social Health Workers (ASHAs) with a mobile phone-based multimedia job aid called mSakhi in 2012.


## Enhanced RAG Pipeline Features

In [39]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Tell me about Inequality and Ideational Politics", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Tell me about Inequality and Ideational Politics'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: An increase in inequality will result in an overall increase in ideational politics, including its individual components - identity politics and worldview politics. This is because higher inequality leads to greater incentives for political parties to engage in ideational politics, particularly identity politics, to drive a wedge within their coalition of voters and justify prevailing high inequality.
Sources: [{'source': 'ideas_interests-2024-05v3.pdf', 'page': 13, 'score': 0.3751818537712097, 'preview': 'To summarise, an increase in inequality:\n(i) will result in an overall increase in ideational politics as well as its individual\ncomponents - be it identity politics or worldview politics;\n(ii) will result in greater identity politics by both political parties with: (a) the party\nrepresenting the ec...'}, {'source': 'ideas_interests-2024-05v3.pdf', 'page': 13, 'score': 0.3751818537712097, 'p

### More Advanced RAG Features

In [40]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Tell me about The Manthan Project in Uttar Pradesh", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Tell me about The Manthan Project in Uttar Pradesh'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
McKenzie ( 2017) finds that the program attracted entrepre -
neurs who invest in higher innovation, resulting in higher 
sales and profits, even during periods of economic crises 
(e.g., in 2016).
• Higher job creation: the competition had a positive impact 
on individuals starting new businesses as well as those 
growing their existing business. Notably, winning firms 
were also more likely to have 10 or more workers. Overall, 
the competition created 7027 jobs (McKenzie 2017 ).
4.3   |   The Manthan Project in Uttar Pradesh, India
In 2012, the Manthan Project provided Accredited Social Health 
Workers (popularly known as ASHA workers or ASHAs) in two 
districts of the state of Uttar Pradesh, India, a mobile phone- based 
multimedia job aid called mSakhi.5 This project was initiated as 
a collaboration between the Bill